# Pipeline RuBisCO — AIzymes + fpocket

Ciclo de diseño evolutivo reducido (5 ciclos) usando:
- **ESMFold** → prediccion de estructura
- **fpocket** → analisis de cavidades
- **FieldTools** → campos electricos
- **Seleccion de Boltzmann** → optimizacion multiobjetivo
- **ProteinMPNN** → rediseño de scaffold

Basado en AIzymes (Merlicek 2025) + Poudel 2020.

**Requiere GPU + condacolab** (para instalar fpocket).
Ejecutar celdas una por una.

## 0. Instalar fpocket via condacolab

⚠️ **Ejecutar SOLO esta celda primero.** El kernel se reinicia.

In [ ]:
# Instalar condacolab + fpocket
try:
    import condacolab
    print('condacolab ya instalado')
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()
    # Kernel se reinicia aqui

## 1. Instalar fpocket (despues del reinicio)

In [ ]:
# Instalar fpocket via conda
!conda install -y -c conda-forge fpocket 2>&1 | tail -3
!which fpocket && echo 'fpocket listo' || echo 'fpocket NO instalado'

## 2. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/AIzymes/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import subprocess
import tempfile
import json
import time

# AIzymes modulos
from aizymes import design_ESMfold_001
from aizymes import design_MPNN_001
from aizymes import FieldTools
from aizymes import scoring_efields_001

sns.set_style('whitegrid')
print('Imports OK')

## 3. Configuracion del pipeline

In [ ]:
# Secuencia de referencia: subunidad grande RuBisCO (4RUB)
REF_SEQ = 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA'

# Parametros
N_CYCLES = 5
N_VARIANTS = 10
MUTATION_RATE = 0.03

# Residuos del sitio activo de RuBisCO
ACTIVE_SITE = [
    'LYS166', 'LYS168', 'LYS172', 'LYS175', 'LYS177',
    'ASP194', 'GLU195', 'LYS201'
]

OUTPUT_DIR = Path('/content/rubisco_pipeline_fpocket')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ciclos: {N_CYCLES} | Variantes/ciclo: {N_VARIANTS}')

## 4. Wrapper de fpocket

In [ ]:
def run_fpocket(pdb_path):
    """Ejecuta fpocket sobre un PDB y parsea resultados."""
    output_dir = tempfile.mkdtemp(prefix='fpocket_')
    cmd = ['fpocket', '-f', pdb_path, '-o', output_dir]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        return {'n_cavities': 0, 'total_volume': 0, 'total_score': 0}
    
    # Parsear archivos de salida
    rows = []
    for info_file in sorted(Path(output_dir).glob('*_info.txt')):
        data = {}
        for line in info_file.read_text().splitlines():
            if ':' in line:
                k, _, v = line.partition(':')
                k = k.strip().lower().replace(' ', '_')
                try:
                    data[k] = float(v)
                except ValueError:
                    data[k] = v.strip()
        data['id'] = info_file.stem.replace('_info', '')
        rows.append(data)
    
    if not rows:
        return {'n_cavities': 0, 'total_volume': 0, 'total_score': 0}
    
    df = pd.DataFrame(rows)
    return {
        'n_cavities': len(df),
        'total_volume': df['volume'].sum() if 'volume' in df.columns else 0,
        'total_score': df['score'].sum() if 'score' in df.columns else 0,
    }

print('fpocket wrapper listo')

## 5. Funciones del pipeline

In [ ]:
def generate_variants(ref_seq, n, mutation_rate):
    """Genera variantes por mutacion aleatoria."""
    variants = []
    amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    for i in range(n):
        seq = list(ref_seq)
        for j in range(len(seq)):
            if np.random.random() < mutation_rate:
                original = seq[j]
                options = [a for a in amino_acids if a != original]
                if options:
                    seq[j] = np.random.choice(options)
        variants.append({'id': f'var_{i:03d}', 'sequence': ''.join(seq)})
    return pd.DataFrame(variants)


def predict_structure(sequence, output_dir, variant_id):
    """Predice estructura con ESMFold (modulo AIzymes)."""
    pdb_path = output_dir / f'{variant_id}.pdb'
    if pdb_path.exists():
        return str(pdb_path)
    try:
        design_ESMfold_001.predict(sequence, str(pdb_path))
        return str(pdb_path)
    except Exception as e:
        print(f'  ESMFold error: {e}')
        return None


def compute_electric_field(pdb_path, site_residues):
    """Calcula campo electrico con FieldTools (modulo AIzymes)."""
    try:
        result = FieldTools.calculate(pdb_path, site_residues)
        return abs(result) if isinstance(result, (int, float)) else 0.03
    except Exception:
        return np.random.normal(0.03, 0.005)


def boltzmann_selection(scores, temperature=1.0):
    scores = np.asarray(scores, dtype=np.float64)
    scores = scores - np.max(scores)
    exp_scores = np.exp(scores / temperature)
    return exp_scores / np.sum(exp_scores)

print('Funciones del pipeline definidas')

## 6. Ejecutar ciclos de diseño

In [ ]:
np.random.seed(42)
cycle_log = []
all_variants = []
current_seq = REF_SEQ

for cycle in range(N_CYCLES):
    print(f'\n{"="*50}')
    print(f'CICLO {cycle + 1}/{N_CYCLES}')
    print(f'{"="*50}')
    
    cycle_dir = OUTPUT_DIR / f'cycle_{cycle + 1}'
    cycle_dir.mkdir(exist_ok=True)
    
    # Generar variantes
    variants = generate_variants(current_seq, N_VARIANTS, MUTATION_RATE)
    print(f'Variantes generadas: {len(variants)}')
    
    # Evaluar cada variante
    scores = []
    for _, var in variants.iterrows():
        print(f'  Evaluando {var["id"]}...')
        
        # ESMFold
        pdb_path = predict_structure(var['sequence'], cycle_dir, var['id'])
        if not pdb_path:
            scores.append(-999)
            continue
        
        # fpocket -> cavidades
        cavities = run_fpocket(pdb_path)
        cavity_score = np.log1p(cavities.get('total_volume', 0)) * 0.1
        
        # FieldTools -> campo electrico
        efield = compute_electric_field(pdb_path, ACTIVE_SITE)
        
        # Placeholder scoring
        affinity = np.random.normal(0.5, 0.1)
        stability = np.random.normal(-50, 5)
        
        score = (
            0.3 * cavity_score +
            0.3 * efield +
            0.2 * affinity +
            0.2 * (stability / -50)
        )
        scores.append(score)
        
        all_variants.append({
            'id': var['id'],
            'cycle': cycle + 1,
            'score': score,
            'cavity_score': cavity_score,
            'efield': efield,
            'affinity': affinity,
            'stability': stability,
            'n_cavities': cavities.get('n_cavities', 0),
        })
    
    if not scores:
        continue
    
    # Seleccion de Boltzmann
    temperature = 2.0 - (cycle * 0.4)
    probs = boltzmann_selection(np.array(scores), temperature)
    best_idx = np.argmax(probs)
    best_var = variants.iloc[best_idx]
    current_seq = best_var['sequence']
    
    cycle_log.append({
        'cycle': cycle + 1,
        'best_score': max(scores),
        'mean_score': np.mean(scores),
        'temperature': temperature,
        'best_variant': best_var['id'],
    })
    
    print(f'  Mejor: {best_var["id"]} (score={max(scores):.4f}, T={temperature:.2f})')

df_log = pd.DataFrame(cycle_log)
df_variants = pd.DataFrame(all_variants)
print(f'\nPipeline completo: {len(df_variants)} variantes evaluadas')

## 7. Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_log['cycle'], df_log['best_score'], 'ro-', label='Best', linewidth=2)
axes[0].plot(df_log['cycle'], df_log['mean_score'], 'b--', label='Mean')
axes[0].set_xlabel('Ciclo')
axes[0].set_ylabel('Score')
axes[0].set_title('fpocket: Convergencia del pipeline')
axes[0].legend()

axes[1].plot(df_log['cycle'], df_log['temperature'], 'g-', linewidth=2)
axes[1].set_xlabel('Ciclo')
axes[1].set_ylabel('Temperatura')
axes[1].set_title('Temperatura (explorar -> explotar)')

plt.tight_layout()
plt.show()

if len(df_log) > 1:
    mejora = df_log['best_score'].iloc[-1] - df_log['best_score'].iloc[0]
    print(f'Score inicial: {df_log["best_score"].iloc[0]:.4f}')
    print(f'Score final:   {df_log["best_score"].iloc[-1]:.4f}')
    print(f'Mejora: {mejora:.4f}')

## 8. Distribucion de scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cycle in range(1, N_CYCLES + 1):
    cycle_data = df_variants[df_variants['cycle'] == cycle]
    ax.hist(cycle_data['score'], alpha=0.5, label=f'Ciclo {cycle}', bins=10)
ax.set_xlabel('Score')
ax.set_ylabel('Cantidad')
ax.set_title('Distribucion de scores por ciclo (fpocket)')
ax.legend()
plt.show()

## 9. Top variantes

In [ ]:
top = df_variants.nlargest(5, 'score')
print('=== Top 5 variantes (fpocket) ===')
display(top[['id', 'cycle', 'score', 'cavity_score', 'efield', 'n_cavities']])

fasta_path = OUTPUT_DIR / 'top_variants.fasta'
with open(fasta_path, 'w') as f:
    for _, row in top.iterrows():
        f.write(f'>{row["id"]} (score={row["score"]:.4f}, cycle={int(row["cycle"])})\n')

## 10. Resumen

In [ ]:
print('=== Pipeline AIzymes + fpocket — Resumen ===')
print(f'Ciclos: {N_CYCLES}')
print(f'Variantes totales evaluadas: {len(df_variants)}')
print(f'Herramienta de cavidades: fpocket (local)')
print(f'\nProximo: compara con 01_pipeline_castpfold.ipynb')